In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN


In [25]:
# 1. Load dataset
data_path = 'dataset/covid_19_indonesia_time_series_all.csv'
df = pd.read_csv(data_path)

# Menampilkan lima baris pertama dataset
df.head()


,Date,Location ISO Code,Location,New Cases,New Deaths,New Recovered,New Active Cases,Total Cases,Total Deaths,Total Recovered,...,Latitude,New Cases per Million,Total Cases per Million,New Deaths per Million,Total Deaths per Million,Total Deaths per 100rb,Case Fatality Rate,Case Recovered Rate,Growth Factor of New Cases,Growth Factor of New Deaths
0,3/1/2020,ID-JK,DKI Jakarta,2,0,0,2,39,20,75,...,-6.204699,0.18,3.60,0.0,1.84,0.18,51.28%,192.31%,NaN,NaN
1,3/2/2020,ID-JK,DKI Jakarta,2,0,0,2,41,20,75,...,-6.204699,0.18,3.78,0.0,1.84,0.18,48.78%,182.93%,1.0,1.0
2,3/2/2020,IDN,Indonesia,2,0,0,2,2,0,0,...,-0.789275,0.01,0.01,0.0,0.00,0.00,0.00%,0.00%,NaN,NaN
3,3/2/2020,ID-RI,Riau,1,0,0,1,1,0,1,...,0.511648,0.16,0.16,0.0,0.00,0.00,0.00%,100.00%,NaN,NaN
4,3/3/2020,ID-JK,DKI Jakarta,2,0,0,2,43,20,75,...,-6.204699,0.18,3.96,0.0,1.84,0.18,46.51%,174.42%,1.0,1.0


In [26]:
# 2. Preprocess data
# Memilih fitur yang digunakan dan mengganti nilai kosong dengan 0
features = ['Total Cases', 'Total Recovered', 'Total Deaths']
X = df[features].fillna(0)

# Menampilkan data hasil preprocessing
X.head()


,Total Cases,Total Recovered,Total Deaths
0,39,75,20
1,41,75,20
2,2,0,0
3,1,1,0
4,43,75,20


In [27]:
# 3. Standardize features
# Menyamakan skala setiap fitur agar perhitungan jarak DBSCAN tidak bias
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Ukuran data setelah standardisasi: {X_scaled.shape}')


Ukuran data setelah standardisasi: (31822, 3)


In [28]:
# 4. DBSCAN Clustering dengan target 3 klaster
# Mencari nilai eps yang menghasilkan tepat 3 klaster, tidak termasuk noise
target_clusters = 3
min_samples_value = 5
eps_values = np.linspace(0.1, 1.0, 10)
best_eps = None

for eps in eps_values:
    labels_tmp = DBSCAN(
        eps=eps,
        min_samples=min_samples_value
    ).fit_predict(X_scaled)
    n_clusters = len(set(labels_tmp)) - (1 if -1 in labels_tmp else 0)

    if n_clusters == target_clusters:
        best_eps = eps
        labels = labels_tmp
        break

if best_eps is None:
    print(
        f'Tidak ditemukan eps antara {eps_values[0]:.1f}–'
        f'{eps_values[-1]:.1f} yang menghasilkan {target_clusters} klaster.'
    )
    best_eps = 0.5
    labels = DBSCAN(
        eps=best_eps,
        min_samples=min_samples_value
    ).fit_predict(X_scaled)
else:
    print(
        f'Ditemukan eps = {best_eps:.2f} yang menghasilkan '
        f'{target_clusters} klaster.'
    )

# Menyimpan label dan menghitung jumlah klaster akhir
df['Cluster_DB'] = labels
unique_labels = set(labels)
num_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
print(
    f'DBSCAN menghasilkan {num_clusters} klaster '
    f'(tidak termasuk noise) dengan eps={best_eps:.2f}.'
)


Ditemukan eps = 0.10 yang menghasilkan 3 klaster.
DBSCAN menghasilkan 3 klaster (tidak termasuk noise) dengan eps=0.10.


In [ ]:
# 5. Visualisasi cluster DBSCAN menggunakan PCA
# PCA mereduksi tiga fitur menjadi dua dimensi agar pola klaster dapat divisualisasikan
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for lbl in sorted(unique_labels):
    idx = labels == lbl
    label_name = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    marker = 'x' if lbl == -1 else 'o'
    color = 'red' if lbl == -1 else ('blue' if lbl == 2 else None)
    plt.scatter(
        components[idx, 0],
        components[idx, 1],
        label=label_name,
        s=50,
        marker=marker,
        color=color
    )

plt.title('DBSCAN Clustering (PCA Projection)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()

# Debug: menjelaskan informasi yang ditampilkan pada grafik
print("\nPenjelasan visualisasi:")
print(f"- PC1 menjelaskan {pca.explained_variance_ratio_[0] * 100:.2f}% variasi data.")
print(f"- PC2 menjelaskan {pca.explained_variance_ratio_[1] * 100:.2f}% variasi data.")
print(f"- Total informasi yang dipertahankan: {pca.explained_variance_ratio_.sum() * 100:.2f}%.")
print("- Titik berdekatan memiliki karakteristik kasus COVID-19 yang mirip.")
print("- Warna berbeda menunjukkan klaster berbeda; tanda x menunjukkan noise DBSCAN.")
for lbl in sorted(unique_labels):
    jumlah = int(np.sum(labels == lbl))
    nama = 'Noise' if lbl == -1 else f'Cluster {lbl}'
    print(f"- {nama}: {jumlah} data.")


In [30]:
# 6. Ringkasan klaster (mean nilai fitur), kecuali noise
cluster_summary = df[df['Cluster_DB'] != -1].groupby('Cluster_DB')[features].mean()
print("DBSCAN Cluster Summary (mean feature values, excluding noise):")
print(cluster_summary)

DBSCAN Cluster Summary (mean feature values, excluding noise):
             Total Cases  Total Recovered   Total Deaths
Cluster_DB                                              
0           9.127901e+04     8.469428e+04    2621.769471
1           4.237727e+06     4.057697e+06  142405.969325
2           6.129455e+06     5.931794e+06  156520.315217


In [34]:
# 7. Evaluasi hasil clustering DBSCAN dengan tiga metode
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# Noise (-1) tidak dinilai karena bukan anggota klaster
mask_cluster = labels != -1
X_evaluasi = X_scaled[mask_cluster]
label_evaluasi = labels[mask_cluster]
jumlah_cluster_evaluasi = len(set(label_evaluasi))

if jumlah_cluster_evaluasi < 2:
    print('Evaluasi tidak dapat dilakukan: DBSCAN menghasilkan kurang dari 2 klaster.')
else:
    silhouette = silhouette_score(X_evaluasi, label_evaluasi)
    davies_bouldin = davies_bouldin_score(X_evaluasi, label_evaluasi)
    calinski_harabasz = calinski_harabasz_score(X_evaluasi, label_evaluasi)

    hasil_evaluasi = pd.DataFrame({
        'Metode Evaluasi': [
            'Silhouette Score',
            'Davies-Bouldin Index',
            'Calinski-Harabasz Index'
        ],
        'Nilai': [silhouette, davies_bouldin, calinski_harabasz],
        'Kriteria Terbaik': [
            'Mendekati 1',
            'Semakin kecil',
            'Semakin besar'
        ]
    })
    display(hasil_evaluasi.round(4))

    print('=== PENJELASAN TIGA METODE EVALUASI ===')
    print(
        f'1. Silhouette Score = {silhouette:.4f}. '
        'Mengukur kekompakan dan pemisahan klaster; nilai mendekati 1 lebih baik.'
    )
    print(
        f'2. Davies-Bouldin Index = {davies_bouldin:.4f}. '
        'Mengukur kemiripan antarklaster; nilai lebih kecil lebih baik.'
    )
    print(
        f'3. Calinski-Harabasz Index = {calinski_harabasz:.4f}. '
        'Membandingkan variasi antarklaster dengan variasi dalam klaster; '
        'nilai lebih besar lebih baik.'
    )
    print(
        f'Evaluasi memakai {len(X_evaluasi)} data dalam '
        f'{jumlah_cluster_evaluasi} klaster; {np.sum(~mask_cluster)} noise dikecualikan.'
    )

# Inertia tidak digunakan karena DBSCAN tidak memiliki centroid seperti K-Means.


,Metode Evaluasi,Nilai,Kriteria Terbaik
0,Silhouette Score,0.9640,Mendekati 1
1,Davies-Bouldin Index,0.0776,Semakin kecil
2,Calinski-Harabasz Index,102051.9546,Semakin besar


=== PENJELASAN TIGA METODE EVALUASI ===
1. Silhouette Score = 0.9640. Mengukur kekompakan dan pemisahan klaster; nilai mendekati 1 lebih baik.
2. Davies-Bouldin Index = 0.0776. Mengukur kemiripan antarklaster; nilai lebih kecil lebih baik.
3. Calinski-Harabasz Index = 102051.9546. Membandingkan variasi antarklaster dengan variasi dalam klaster; nilai lebih besar lebih baik.
Evaluasi memakai 31727 data dalam 3 klaster; 95 noise dikecualikan.
